#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Count-based Word Embeddings (Inglés)

# Pre-Processing

In [1]:
#Importar Librerías
import spacy
import pandas as pd
import numpy as np

In [2]:
#Cargar Data
dataset = pd.read_csv('/content/pubmed_bow_model.csv')

In [3]:
#Revisar Dataset
print("Dataset Columns:", dataset.columns)
print("Volumetría Dataset:", dataset.shape)

Dataset Columns: Index(['Title'], dtype='object')
Volumetría Dataset: (1000, 1)


In [4]:
#Revisar Dataset
dataset.head()

,Title
0,Expression of p53 and coexistence of HPV in pr...
1,Vitamin D status in pregnant Indian women acro...
2,[Identification of a functionally important di...
3,Multilayer capsules: a promising microencapsul...
4,"Nanohydrogel with N,N'-bis(acryloyl)cystine cr..."


In [5]:
#Descargar Pipeline (English)
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
#Cargar Pipeline Model
nlp = spacy.load("en_core_web_sm")

In [7]:
#Función para Pre-Processing + Tokenización
def get_tokens(text):
  doc = nlp(text)
  tokens = [token.lemma_.lower() for token in doc if token if not token.is_stop and not token.is_punct]
  return tokens

# Modelo BoW (TF-IDF Scores)

In [8]:
#Importar Librerías
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
#Crear Objeto CountVectorizer
tfidf_vectorizer = TfidfVectorizer(lowercase=False,
                                   preprocessor=None,
                                   tokenizer=get_tokens)

In [10]:
#Estadísticas Descriptivas
tfidf_vectorizer.fit(dataset['Title'])
print("Tamaño Vocabulario:", len(tfidf_vectorizer.get_feature_names_out()))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Tamaño Vocabulario: 4547


In [11]:
#Generar Representación Vectorial
data_tfidf_vectorized = tfidf_vectorizer.transform(dataset['Title'])
print("Dimensionalidad Rep. Vectorial:", data_tfidf_vectorized.shape)

Dimensionalidad Rep. Vectorial: (1000, 4547)


# Latent Semantic Analysis (Reducción Dimensionalidad)

In [12]:
#Importar Librerías
from sklearn.decomposition import TruncatedSVD

In [13]:
#Parametrización & Ajuste Objeto SVD
n_components = 750
SVD = TruncatedSVD(n_components=n_components,
                   random_state=0)
SVD = SVD.fit(data_tfidf_vectorized)

In [14]:
#Resumen Varianza Explicada
print("Total Varianza Explicada:", round(SVD.explained_variance_ratio_.sum(),3))

Total Varianza Explicada: 0.879


In [15]:
#Transformación Matriz Rep. Vectorial
data_tfidf_lsa = SVD.transform(data_tfidf_vectorized)
print("Dimensionalidad Rep. Vectorial:", data_tfidf_lsa.shape)

Dimensionalidad Rep. Vectorial: (1000, 750)
